In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
conn = sqlite3.connect('../data/processed/telemetry.db')

tables = pd.read_sql("PRAGMA table_list;", conn)

for idx, row in tables.iterrows():
    print(f"{idx}: {row['name']} ({row['ncol']} columns)")

0: function_durations_percentiles.anon.d09 (14 columns)
1: function_durations_percentiles.anon.d08 (14 columns)
2: app_memory_percentiles.anon.d09 (12 columns)
3: app_memory_percentiles.anon.d08 (12 columns)
4: invocations_per_function_md.anon.d03 (1444 columns)
5: app_memory_percentiles.anon.d01 (12 columns)
6: invocations_per_function_md.anon.d02 (1444 columns)
7: app_memory_percentiles.anon.d07 (12 columns)
8: function_durations_percentiles.anon.d02 (14 columns)
9: invocations_per_function_md.anon.d05 (1444 columns)
10: function_durations_percentiles.anon.d11 (14 columns)
11: invocations_per_function_md.anon.d14 (1444 columns)
12: function_durations_percentiles.anon.d10 (14 columns)
13: invocations_per_function_md.anon.d13 (1444 columns)
14: function_durations_percentiles.anon.d01 (14 columns)
15: invocations_per_function_md.anon.d04 (1444 columns)
16: app_memory_percentiles.anon.d06 (12 columns)
17: function_durations_percentiles.anon.d13 (14 columns)
18: function_durations_percent

In [3]:
table_name = '"function_durations_percentiles.anon.d04"' 
schema_info = pd.read_sql(f"PRAGMA table_info({table_name});", conn)
print(f"\nSchema for {table_name}:")
print(schema_info[['name', 'type']])


Schema for "function_durations_percentiles.anon.d04":
                      name     type
0                hashowner     TEXT
1                  hashapp     TEXT
2             hashfunction     TEXT
3                  average  INTEGER
4                    count  INTEGER
5                  minimum     REAL
6                  maximum     REAL
7     percentile_average_0  INTEGER
8     percentile_average_1  INTEGER
9    percentile_average_25  INTEGER
10   percentile_average_50  INTEGER
11   percentile_average_75  INTEGER
12   percentile_average_99  INTEGER
13  percentile_average_100  INTEGER


In [4]:
table_name = '"app_memory_percentiles.anon.d09"' 
schema_info = pd.read_sql(f"PRAGMA table_info({table_name});", conn)
print(f"\nSchema for {table_name}:")
print(schema_info[['name', 'type']])


Schema for "app_memory_percentiles.anon.d09":
                         name     type
0                   hashowner     TEXT
1                     hashapp     TEXT
2                 samplecount  INTEGER
3          averageallocatedmb  INTEGER
4     averageallocatedmb_pct1  INTEGER
5     averageallocatedmb_pct5  INTEGER
6    averageallocatedmb_pct25  INTEGER
7    averageallocatedmb_pct50  INTEGER
8    averageallocatedmb_pct75  INTEGER
9    averageallocatedmb_pct95  INTEGER
10   averageallocatedmb_pct99  INTEGER
11  averageallocatedmb_pct100  INTEGER


In [5]:
table_name = '"invocations_per_function_md.anon.d04"' 
schema_info = pd.read_sql(f"PRAGMA table_info({table_name});", conn)
print(f"\nSchema for {table_name}:")
print(schema_info[['name', 'type']])


Schema for "invocations_per_function_md.anon.d04":
              name     type
0        hashowner     TEXT
1          hashapp     TEXT
2     hashfunction     TEXT
3          trigger     TEXT
4                1  INTEGER
...            ...      ...
1439          1436  INTEGER
1440          1437  INTEGER
1441          1438  INTEGER
1442          1439  INTEGER
1443          1440  INTEGER

[1444 rows x 2 columns]


In [6]:
query_burst = """
SELECT 
    hashapp, 
    AVG(averageallocatedmb) as avg_mem, 
    MAX(averageallocatedmb) as peak_mem,
    (MAX(averageallocatedmb) * 1.0 / NULLIF(AVG(averageallocatedmb), 0)) as burst_factor
FROM "app_memory_percentiles.anon.d01"
GROUP BY hashapp
HAVING avg_mem > 5  -- Ignore tiny apps that aren't doing anything
ORDER BY burst_factor DESC
LIMIT 10;
"""

df_burst = pd.read_sql(query_burst, conn)
print("Top 10 Most 'Bursty' Apps (Potential Noisy Neighbors):")
display(df_burst)

Top 10 Most 'Bursty' Apps (Potential Noisy Neighbors):


,hashapp,avg_mem,peak_mem,burst_factor
0,0d1695cb8bba4b0e0889973c7f579b2d60381bd3565daf...,133.0,167,1.255639
1,a74f6f74cb944ea73117f4156c4e96e8473ac1d513e042...,146.5,179,1.221843
2,d780c03a2a32599ad5d4590b79f2771d4bd65efb3acf53...,142.0,169,1.190141
3,7d5668e85bd02eedf7a6c50d79557f51caca66f2da9fe7...,149.5,174,1.163880
4,2683c4571019fd3c79c201d4280b160d0538161ec4109e...,144.0,167,1.159722
5,f5c26057cca77d69416a5e0f705b188c929bf911667237...,125.5,142,1.131474
6,3a8cdc05ab3996128db0ecb34d1d7f955a61570fe35705...,175.0,184,1.051429
7,2c9d52af8baafe90533bb3e71482e80bcf9266ac25c445...,175.0,182,1.040000
8,e865e772b5eebf8dd3d8df3f1ca4b6fa40a7ef25a8a110...,182.5,186,1.019178
9,a197cb21ed60f11ab11c76ca39891badfb39f975db830b...,189.5,190,1.002639


In [7]:
query_cold_start = """
SELECT 
    i.hashapp, 
    i.hashfunction,
    (i."1" + i."2" + i."720" + i."1440") as sample_invocations, -- Simplified sum for demo
    d.average as avg_latency,
    d.percentile_average_99 as tail_latency
FROM "invocations_per_function_md.anon.d04" i
JOIN "function_durations_percentiles.anon.d04" d 
    ON i.hashapp = d.hashapp AND i.hashfunction = d.hashfunction
WHERE sample_invocations > 0
ORDER BY tail_latency DESC
LIMIT 10;
"""

df_latencies = pd.read_sql(query_cold_start, conn)
print("\nLatency vs. Demand (Looking for Cold Starts):")
display(df_latencies)


Latency vs. Demand (Looking for Cold Starts):


,hashapp,hashfunction,sample_invocations,avg_latency,tail_latency
0,4b05f738540108d0269b700201cb41e3999dec14099a47...,398ff94a81a6eea8e999fb0f5831caa31e861d6426e7e3...,7,646965,4622582
1,565472b1588aaa568a2256cba0d415dc3620eade1d1ec3...,42bc5fa1eeab627be77ff919b76d0bbd9eeaf22be233c1...,2,48877,2333919
2,ee2f53f89a35c322a948262c5b545eb46df8ca96d47754...,1e5bfcc662ff982b62ca347d1058f43780275234da00b4...,1,785963,912818
3,4f48a66075e643f6ad6807799b9d07a68018088fb29dcd...,c0ac9ff19c999689d992e1872edb840302ebd460f3127c...,14,182882,758822
4,b5c397e3357914a243d2fc63a4bfce4666989eae186f72...,acf65d6a09b51b8981f09e75d67e136995215a7619ab49...,1,137497,750523
5,60393a5c156c1905114e7104d51cff90fbf6ca7068c964...,a0ad156e89a6138c4c73adb3306e6beb90782a3ab059fd...,1,334269,705600
6,1e99c3ba66135a2606479d83fbc95d4e9695e7cb4464dc...,7204b7d910ee9a83e5d35a25366e516a1bc1a651190280...,2,232142,698803
7,9cd60b5bc4503e77a8e92d9261e6466452e64e02a4eb30...,d3209b753752c8641cde05ba81b3ceb0d175b1f986fee7...,1,667216,667216
8,d497aff4f22e35f51b996f7a85e34bab98337fdf63e39b...,c961c5fd78368ed4d5966a2eacc9661e1cca0012a5f173...,1,650296,650296
9,68ffaeb16df7f9b61e4ec98376d9e1f514d4f3fe7682d0...,bf042d71605a7b43e25d16a65c98c6ea8f506de65aa186...,5,119109,641822


In [11]:
query = """
SELECT *
FROM "function_durations_percentiles.anon.d01"
LIMIT 5;
"""

df_query = pd.read_sql(query, conn)
display(df_query)

,hashowner,hashapp,hashfunction,average,count,minimum,maximum,percentile_average_0,percentile_average_1,percentile_average_25,percentile_average_50,percentile_average_75,percentile_average_99,percentile_average_100
0,5640c1597ef75fa9a7e9c6925022a039a4ba9829241709...,5126901eff078c9a1f5295c859c9327588284a43cc2c0b...,c968871b4ef0123401975d026b85cae2ad7dad9d06ae94...,100,2880,0.0,2595.0,0,0,0,1,1,1376,2595
1,5640c1597ef75fa9a7e9c6925022a039a4ba9829241709...,5126901eff078c9a1f5295c859c9327588284a43cc2c0b...,1bc2d86badd21b18a8533d8e961e52585e5ad1fee0e2c2...,202,365,0.0,2596.0,0,0,1,1,6,2326,2596
2,41d6e09d0f86f5aaa1df842c1ac4f14fbd4dc676bec7e0...,77a93348150f5281c32c9027870a77983ad6eba72e6c28...,b4d7cb9985cd8c1b7fe5ef888bbde1883929e9a5b5d2a6...,11,37815,1.0,3913.0,1,1,3,5,8,155,3146
3,e117b8c1676e11d859824b18fc0202ae8eaa89cb3f58e2...,5c84cc1fbc4261f22d6f626304670979c1f1b94709acc8...,ff612aae7b380ef81ccef063ff814354ee16018af65dc1...,0,34466,0.0,123.0,0,0,0,0,0,1,10
4,3c8c13c6bd162490dae4402bcb9ff1e5c6c14bb21da35a...,3b80dfff065220947f7d626dd6c9176731d3e17591e496...,ee79840da82525548358f4b91d9fa1e550ec7af36bbe3e...,389,288,78.0,5607.0,78,78,109,136,180,4761,5607


In [12]:
query = """
SELECT *
FROM "app_memory_percentiles.anon.d01"
LIMIT 5;
"""

df_query = pd.read_sql(query, conn)
display(df_query)

,hashowner,hashapp,samplecount,averageallocatedmb,averageallocatedmb_pct1,averageallocatedmb_pct5,averageallocatedmb_pct25,averageallocatedmb_pct50,averageallocatedmb_pct75,averageallocatedmb_pct95,averageallocatedmb_pct99,averageallocatedmb_pct100
0,71ca12c7af70d021e285b51b245942f8432df6463ff9f2...,7ca324d9fc836a5d4562811c11ce3719530ee919dd1fb9...,17235,108,108,108,108,109,109,109,110,110
1,dca6b75d51b494146b50f93b0f1a863ce99164942695e8...,cfccc67d83c3c4dc1dd075e77e41232282e4d7a75d2ba9...,536284,322,167,199,251,307,357,532,692,991
2,e564f454ffe6e39c133b43183ca5c1537fec3a55b3a73c...,93db1e4c30e99d1c416ee453dd67504641b88144e23219...,34726,673,175,301,471,679,875,1036,1179,1288
3,15d422b003e198041546b5a1b46aa4d3aaada57de592a8...,e59cebb1c16975c6ee9dd52430c55380dc305ae1fb9171...,17238,95,95,95,96,96,96,96,96,96
4,f5b5e5b8537dd76458e850d7c0dc4057dcd8e0dfa50d36...,fb68374027828434785f3c5ce09fcdc2e44212559643bd...,19927,204,182,185,202,206,211,214,223,292


In [19]:
query = """
SELECT *
FROM "invocations_per_function_md.anon.d01"
LIMIT 5;
"""

df_query = pd.read_sql(query, conn)
display(df_query)

,hashowner,hashapp,hashfunction,trigger,1,2,3,4,5,6,...,1431,1432,1433,1434,1435,1436,1437,1438,1439,1440
0,71ca12c7af70d021e285b51b245942f8432df6463ff9f2...,7ca324d9fc836a5d4562811c11ce3719530ee919dd1fb9...,520dbd6bd906840012aa0c4b778743efc7c0ac7b7caf96...,http,0,0,0,1,0,0,...,0,0,0,1,0,0,0,0,1,0
1,71ca12c7af70d021e285b51b245942f8432df6463ff9f2...,0d0ac65651f54ae3285a59564d64e39238b516fa1d5b56...,115ca7a2b5bc290052c3da74cd0347d19c3c67b7d5aa66...,http,0,0,0,1,0,0,...,0,0,0,1,0,0,0,0,1,0
2,71ca12c7af70d021e285b51b245942f8432df6463ff9f2...,a04487a6ba1e14296eb7647e4963180d28bef7a90a8fc5...,93e6c664773bbec3a7f50a0e92fa7e97401a802dc6eed8...,orchestration,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,71ca12c7af70d021e285b51b245942f8432df6463ff9f2...,a04487a6ba1e14296eb7647e4963180d28bef7a90a8fc5...,740c5c767e4b9978ee59a97d1829cfbaf755a47806a311...,http,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,71ca12c7af70d021e285b51b245942f8432df6463ff9f2...,a04487a6ba1e14296eb7647e4963180d28bef7a90a8fc5...,c108b4864b866b38b80d0e4594cc6d038f39668b804a1b...,http,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
